# Banana Leaf Disease Detector — Multi-Label + Negative-Class Pipeline (v7)

This notebook replaces the old single-label (one-of-4-classes) pipeline with **four independent binary models**:

| Model | Question | Trained on |
|---|---|---|
| `leaf_detector` | Is this even a banana leaf? | `SINGLE_LABEL/` + `MULTI_LABEL/` (positive) vs. `NEGATIVE/` (negative) |
| `cordana` | Is Cordana present? | every image whose folder name contains `cordana` |
| `pestalotiopsis` | Is Pestalotiopsis present? | every image whose folder name contains `pestalotiopsis` |
| `sigatoka` | Is Sigatoka present? | every image whose folder name contains `sigatoka` |

Each disease model is a plain yes/no classifier that runs independently of the other two, so a leaf can end up flagged with **any combination** of zero to three diseases — not a single forced label. A leaf detector runs first; if a photo doesn't look like a banana leaf at all, none of the disease models even run.

**Dataset layout expected on Drive** (`bananaleaf_dataset/BananaLeaf_Dataset/`):
```
SINGLE_LABEL/{Cordana,Healthy,Pestalotiopsis,Sigatoka}/*.jpg   (500 each)
MULTI_LABEL/{cordana_pestalotiopsis,cordana_sigatoka,pestalotiopsis_sigatoka,cordana_pestalotiopsis_sigatoka}/*.jpg  (250 each)
NEGATIVE/{background,objects,other_plant,person}/*.jpg  (~212 total)
```


In [ ]:
!pip install -q scikit-image tqdm joblib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil

DRIVE_DIR = "/content/drive/MyDrive/bananaleaf_dataset"
DATASET_ZIP = f"{DRIVE_DIR}/bananaleaf_dataset.zip"   # BananaLeaf_Dataset/ folder zipped
DATASET_DIR = "/content/BananaLeaf_Dataset"

os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.exists(DATASET_DIR):
    with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
        z.extractall("/content")
    # handle a possible extra nesting level (BananaLeaf_Dataset/BananaLeaf_Dataset/...)
    if not os.path.isdir(os.path.join(DATASET_DIR, "SINGLE_LABEL")):
        nested = [d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))]
        if len(nested) == 1:
            inner = os.path.join(DATASET_DIR, nested[0])
            for item in os.listdir(inner):
                shutil.move(os.path.join(inner, item), DATASET_DIR)

print("Dataset ready at", DATASET_DIR)

## 1. Dataset overview

In [ ]:
import glob

def count_images(folder):
    exts = ("*.jpg", "*.jpeg", "*.png")
    return sum(len(glob.glob(os.path.join(folder, e))) for e in exts)

print("SINGLE_LABEL:")
for c in ["Cordana", "Healthy", "Pestalotiopsis", "Sigatoka"]:
    print(f"  {c}: {count_images(os.path.join(DATASET_DIR, 'SINGLE_LABEL', c))}")

print("\nMULTI_LABEL:")
for c in sorted(glob.glob(os.path.join(DATASET_DIR, "MULTI_LABEL", "*"))):
    print(f"  {os.path.basename(c)}: {count_images(c)}")

print("\nNEGATIVE:")
for c in sorted(glob.glob(os.path.join(DATASET_DIR, "NEGATIVE", "*"))):
    print(f"  {os.path.basename(c)}: {count_images(c)}")

## 2. Feature-extraction pipeline

Two feature sets:

1. **Lesion features** (15-dim: HSV color moments + GLCM texture) — computed on a **segmented lesion mask**, feeds the three disease-presence models.
2. **Leaf-detector features** (17-dim: the same 15 lesion-style values + `green_ratio` + `texture_std`) — feeds the leaf/not-leaf gate. Even for non-leaf photos, running the same segmentation pipeline still produces *some* numbers (Otsu splits by brightness, K-means clusters HSV pixels regardless of what's actually in frame) — these numbers just won't look leaf-like, which is exactly the signal the leaf-detector learns to recognize, on top of the more direct green_ratio/texture_std check.

### The multi-cluster fix

The very first version of this pipeline picked only the **single** K-means cluster furthest from healthy green as "the lesion". That's fine for a leaf with one disease, but a leaf with **two** co-occurring diseases (different lesion colors/textures) produces **two** different non-green clusters — picking only one silently discarded whichever one wasn't chosen, starving the multi-label models of exactly the cases they most needed to learn from.

This version instead combines **every** cluster whose center hue is far enough from green (`NON_GREEN_HUE_MARGIN`) into one mask. Verified on this dataset: this raised held-out recall on Cordana and Pestalotiopsis by ~7 percentage points each versus the single-cluster version, with no meaningful precision loss.

In [ ]:
import cv2
import numpy as np
from skimage.feature import graycomatrix, graycoprops

IMG_SIZE = 200
GREEN_HUE = 60
KMEANS_K = 3
NON_GREEN_HUE_MARGIN = 15.0   # cluster counts as "non-healthy tissue" past this hue distance

GLCM_PROPS = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]

LESION_FEATURE_NAMES = (
    ["H_mean", "H_std", "H_skew", "S_mean", "S_std", "S_skew", "V_mean", "V_std", "V_skew"]
    + [f"glcm_{p}" for p in GLCM_PROPS]
)
LEAF_FEATURE_NAMES = LESION_FEATURE_NAMES + ["green_ratio", "texture_std"]
DISEASE_LABELS = ["Cordana", "Pestalotiopsis", "Sigatoka"]   # multi-hot column order


def preprocess(img_bgr, size=IMG_SIZE):
    img = cv2.resize(img_bgr, (size, size), interpolation=cv2.INTER_AREA)
    img = cv2.GaussianBlur(img, (5, 5), 0)
    return img


def segment_leaf_and_lesion(img_bgr, k=KMEANS_K):
    """Otsu (leaf vs background) + K-means on HSV, restricted to the Otsu
    foreground, with ALL non-green clusters combined into one mask."""
    img = preprocess(img_bgr)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    _, otsu_mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if otsu_mask.mean() > 127:
        otsu_mask = cv2.bitwise_not(otsu_mask)
    if otsu_mask.sum() / 255 < 0.05 * img.size / 3:
        otsu_mask = np.full_like(gray, 255)

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    fg_flat = otsu_mask.reshape(-1) > 0
    Z_all = hsv.reshape((-1, 3)).astype(np.float32)
    Z_fg = Z_all[fg_flat]

    lesion_mask = np.zeros((IMG_SIZE * IMG_SIZE,), dtype=np.uint8)

    if len(Z_fg) >= k:
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 0.5)
        _, labels_fg, centers = cv2.kmeans(Z_fg, k, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
        labels_fg = labels_fg.flatten()

        hue_dist = np.abs(centers[:, 0] - GREEN_HUE)
        non_green_clusters = np.where(hue_dist >= NON_GREEN_HUE_MARGIN)[0]
        if len(non_green_clusters) == 0:
            non_green_clusters = [int(np.argmax(hue_dist))]

        cluster_hit = np.isin(labels_fg, non_green_clusters)
        lesion_mask[fg_flat] = np.where(cluster_hit, 255, 0).astype(np.uint8)

    lesion_mask = lesion_mask.reshape(IMG_SIZE, IMG_SIZE)
    mask = cv2.bitwise_and(lesion_mask, otsu_mask)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    if mask.sum() / 255 < 20:
        mask = lesion_mask

    return img, hsv, mask, otsu_mask


def color_moments(hsv_img, mask):
    feats = []
    m = mask > 0
    if m.sum() < 20:
        m = np.ones_like(mask, dtype=bool)
    for ch in range(3):
        chan = hsv_img[:, :, ch][m].astype(np.float64)
        mean = chan.mean()
        std = chan.std()
        third_moment = ((chan - mean) ** 3).mean()
        skew = np.sign(third_moment) * (abs(third_moment) ** (1 / 3))
        feats.extend([mean, std, skew])
    return feats


def glcm_features(bgr_img, mask):
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    m = mask > 0
    if m.sum() < 20:
        m = np.ones_like(mask, dtype=bool)
    roi = np.where(m, gray, 0)
    glcm = graycomatrix(roi, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                         levels=256, symmetric=True, normed=True)
    return [graycoprops(glcm, p).mean() for p in GLCM_PROPS]


def leaf_gate_features(img_bgr):
    """green_ratio + texture_std, matching ImageProcessor.isLikelyLeafPhoto in Kotlin exactly."""
    img = preprocess(img_bgr)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    green_mask = cv2.inRange(hsv, (25, 30, 30), (95, 255, 255))
    green_ratio = cv2.countNonZero(green_mask) / (IMG_SIZE * IMG_SIZE)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    m = green_mask > 0
    texture_std = lap[m].std() if m.sum() > 0 else 0.0
    return [green_ratio, texture_std]


def extract_lesion_features(path):
    """15-dim feature vector for the disease multi-label models."""
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        return None
    img, hsv, mask, _ = segment_leaf_and_lesion(img_bgr)
    return color_moments(hsv, mask) + glcm_features(img, mask)


def extract_leaf_features(path):
    """17-dim feature vector for the leaf/not-leaf gate."""
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        return None
    lesion = extract_lesion_features(path)
    if lesion is None:
        return None
    return lesion + leaf_gate_features(img_bgr)


def multilabel_from_folder(relpath):
    name = relpath.lower()
    return [1 if "cordana" in name else 0,
            1 if "pestalotiopsis" in name else 0,
            1 if "sigatoka" in name else 0]

print("Pipeline ready.")

## 3. Extract features for the disease multi-label dataset

`SINGLE_LABEL` + `MULTI_LABEL` only (2000 + 1000 = 3000 images) — `NEGATIVE` images aren't leaves and have no disease label.

Cached to Drive so a disconnect doesn't cost you the extraction time.

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

DISEASE_FEATURES_CSV = f"{DRIVE_DIR}/features_disease.csv"

if os.path.exists(DISEASE_FEATURES_CSV):
    print(f"Loading cached features from Drive: {DISEASE_FEATURES_CSV}")
    disease_df = pd.read_csv(DISEASE_FEATURES_CSV)
else:
    rows = []
    single_label_dirs = {"Cordana": [1,0,0], "Pestalotiopsis": [0,1,0], "Sigatoka": [0,0,1], "Healthy": [0,0,0]}
    for cls, multihot in single_label_dirs.items():
        folder = os.path.join(DATASET_DIR, "SINGLE_LABEL", cls)
        for p in tqdm(glob.glob(os.path.join(folder, "*")), desc=f"SINGLE_LABEL/{cls}"):
            f = extract_lesion_features(p)
            if f is not None:
                rows.append([p, f"SINGLE_LABEL/{cls}"] + f + multihot)

    for folder in sorted(glob.glob(os.path.join(DATASET_DIR, "MULTI_LABEL", "*"))):
        combo = os.path.basename(folder)
        multihot = multilabel_from_folder(combo)
        for p in tqdm(glob.glob(os.path.join(folder, "*")), desc=f"MULTI_LABEL/{combo}"):
            f = extract_lesion_features(p)
            if f is not None:
                rows.append([p, f"MULTI_LABEL/{combo}"] + f + multihot)

    disease_df = pd.DataFrame(rows, columns=["path", "source_folder"] + LESION_FEATURE_NAMES + DISEASE_LABELS)
    disease_df.to_csv(DISEASE_FEATURES_CSV, index=False)
    print(f"Saved: {DISEASE_FEATURES_CSV}")

print(disease_df.shape)
disease_df.groupby("source_folder").size()

## 4. Extract features for the leaf-detector dataset

All 3212 images: `SINGLE_LABEL` + `MULTI_LABEL` (is_leaf=1) and `NEGATIVE` (is_leaf=0).

In [ ]:
LEAF_FEATURES_CSV = f"{DRIVE_DIR}/features_leaf.csv"

if os.path.exists(LEAF_FEATURES_CSV):
    print(f"Loading cached features from Drive: {LEAF_FEATURES_CSV}")
    leaf_df = pd.read_csv(LEAF_FEATURES_CSV)
else:
    rows = []
    all_leaf_dirs = (
        [(f"SINGLE_LABEL/{c}", 1) for c in ["Cordana", "Healthy", "Pestalotiopsis", "Sigatoka"]]
        + [(f"MULTI_LABEL/{os.path.basename(d)}", 1) for d in sorted(glob.glob(os.path.join(DATASET_DIR, "MULTI_LABEL", "*")))]
        + [(f"NEGATIVE/{os.path.basename(d)}", 0) for d in sorted(glob.glob(os.path.join(DATASET_DIR, "NEGATIVE", "*")))]
    )
    for reldir, is_leaf in all_leaf_dirs:
        folder = os.path.join(DATASET_DIR, reldir)
        for p in tqdm(glob.glob(os.path.join(folder, "*")), desc=reldir):
            f = extract_leaf_features(p)
            if f is not None:
                rows.append([p, reldir, is_leaf] + f)

    leaf_df = pd.DataFrame(rows, columns=["path", "source_folder", "is_leaf"] + LEAF_FEATURE_NAMES)
    leaf_df.to_csv(LEAF_FEATURES_CSV, index=False)
    print(f"Saved: {LEAF_FEATURES_CSV}")

print(leaf_df.shape)
print(leaf_df["is_leaf"].value_counts())

## 5. Train the leaf/not-leaf detector

Stratified 70/15/15 split, `class_weight="balanced"` to handle the ~3000-vs-~210 imbalance, `GridSearchCV` on F1.

A threshold sweep follows — this is the important part. The default 0.5 cutoff isn't necessarily the best one for a task where accepting a non-leaf photo (a false "leaf") matters more than the usual balanced-accuracy story suggests.

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, precision_recall_fscore_support, accuracy_score
import joblib

X = leaf_df[LEAF_FEATURE_NAMES].values
y = leaf_df["is_leaf"].values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

leaf_scaler = StandardScaler().fit(X_train)
X_train_s, X_val_s, X_test_s = leaf_scaler.transform(X_train), leaf_scaler.transform(X_val), leaf_scaler.transform(X_test)
print(f"train={len(X_train)}  val={len(X_val)}  test={len(X_test)}")

rf_param_grid = {"n_estimators": [100, 200, 400], "max_depth": [None, 10, 20], "min_samples_split": [2, 5]}
leaf_grid = GridSearchCV(RandomForestClassifier(random_state=42, class_weight="balanced"),
                          rf_param_grid, cv=5, scoring="f1", n_jobs=-1)
leaf_grid.fit(X_train_s, y_train)
leaf_model = leaf_grid.best_estimator_
print("Best params:", leaf_grid.best_params_)

proba_test = leaf_model.predict_proba(X_test_s)[:, 1]
pred_test = (proba_test >= 0.5).astype(int)
acc = accuracy_score(y_test, pred_test)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, pred_test, average="binary")
print(f"\nTest (t=0.5): acc={acc:.4f} precision={prec:.4f} recall={rec:.4f} f1={f1:.4f}")
print(classification_report(y_test, pred_test, target_names=["Not a leaf", "Leaf"]))

In [ ]:
print("Threshold sweep on TEST set (leaf-probability cutoff for accepting a photo):")
for thresh in [0.3, 0.4, 0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]:
    pred_t = (proba_test >= thresh).astype(int)
    tn = np.sum((pred_t == 0) & (y_test == 0)); fp = np.sum((pred_t == 1) & (y_test == 0))
    fn = np.sum((pred_t == 0) & (y_test == 1)); tp = np.sum((pred_t == 1) & (y_test == 1))
    neg_recall = tn / (tn + fp) if (tn+fp) > 0 else 0   # correctly rejects non-leaf
    pos_recall = tp / (tp + fn) if (tp+fn) > 0 else 0   # correctly accepts real leaf
    print(f"  t={thresh:.2f}: reject-non-leaf={neg_recall:.3f} ({tn}/{tn+fp})  accept-real-leaf={pos_recall:.3f} ({tp}/{tp+fn})")

# t=0.65 chosen for the app: 100% real-leaf acceptance, ~91% non-leaf rejection.
# Only ~32 negative photos were in this test split, so treat the rejection
# rate as a rough estimate -- collect more NEGATIVE/ examples over time and
# retrain to narrow it.
LEAF_THRESHOLD = 0.65

joblib.dump(leaf_scaler, "/content/scaler_leaf.pkl")
joblib.dump(leaf_model, "/content/model_leaf.pkl")
print("\nSaved model_leaf.pkl and scaler_leaf.pkl")

## 6. Train the three disease-presence models

One independent binary `RandomForestClassifier` per disease (not a single multi-class model — that's the whole point: any combination of the three can be flagged at once). Stratified by the exact 8-way `source_folder` combo so every split has proportional representation of every single- and multi-disease case, not just each binary label independently.

In [ ]:
FEATURE_NAMES_15 = LESION_FEATURE_NAMES  # alias for clarity below

X = disease_df[FEATURE_NAMES_15].values
Y = disease_df[DISEASE_LABELS].values
strat_key = disease_df["source_folder"].values

idx = np.arange(len(disease_df))
idx_train, idx_temp = train_test_split(idx, test_size=0.30, stratify=strat_key, random_state=42)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, stratify=strat_key[idx_temp], random_state=42)

X_train, X_val, X_test = X[idx_train], X[idx_val], X[idx_test]
Y_train, Y_val, Y_test = Y[idx_train], Y[idx_val], Y[idx_test]

disease_scaler = StandardScaler().fit(X_train)
X_train_s, X_val_s, X_test_s = disease_scaler.transform(X_train), disease_scaler.transform(X_val), disease_scaler.transform(X_test)
print(f"train={len(X_train)}  val={len(X_val)}  test={len(X_test)}")

disease_models = {}
summary_rows = []

for i, disease in enumerate(DISEASE_LABELS):
    grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_param_grid, cv=5, scoring="f1", n_jobs=-1)
    grid.fit(X_train_s, Y_train[:, i])
    best = grid.best_estimator_
    print(f"\n=== {disease} ===  best params: {grid.best_params_}")

    pred = best.predict(X_test_s)
    acc = accuracy_score(Y_test[:, i], pred)
    prec, rec, f1, _ = precision_recall_fscore_support(Y_test[:, i], pred, average="binary")
    print(f"Test (t=0.5): acc={acc:.4f} precision={prec:.4f} recall={rec:.4f} f1={f1:.4f}")

    disease_models[disease] = best
    summary_rows.append([disease, acc, prec, rec, f1])

summary_df = pd.DataFrame(summary_rows, columns=["disease","accuracy","precision","recall","f1"]).set_index("disease")
print("\n=== Disease presence model summary (t=0.5) ===")
print(summary_df)

### Per-disease threshold tuning

Default 0.5 leaves recall around 0.69-0.72 for each disease — too many real cases missed for a diagnostic aid. Sweep each disease's threshold on the **validation** set (not test, to avoid overfitting the threshold choice), pick whichever maximizes F1, then confirm on the held-out test set.

In [ ]:
chosen_thresholds = {}
for i, disease in enumerate(DISEASE_LABELS):
    model = disease_models[disease]
    proba_val = model.predict_proba(X_val_s)[:, 1]
    best_t, best_f1 = 0.5, -1
    print(f"\n=== {disease}: threshold sweep on VALIDATION set ===")
    for t in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
        pred = (proba_val >= t).astype(int)
        prec, rec, f1, _ = precision_recall_fscore_support(Y_val[:, i], pred, average="binary", zero_division=0)
        flag = ""
        if f1 > best_f1:
            best_f1, best_t = f1, t
            flag = "  <- best F1"
        print(f"  t={t:.2f}: precision={prec:.3f} recall={rec:.3f} f1={f1:.3f}{flag}")
    chosen_thresholds[disease] = best_t

print("\n=== Final TEST-set metrics at chosen thresholds ===")
final_rows = []
for i, disease in enumerate(DISEASE_LABELS):
    model = disease_models[disease]
    proba_test = model.predict_proba(X_test_s)[:, 1]
    t = chosen_thresholds[disease]
    pred = (proba_test >= t).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(Y_test[:, i], pred, average="binary", zero_division=0)
    print(f"{disease} (t={t}): precision={prec:.3f} recall={rec:.3f} f1={f1:.3f}")
    final_rows.append([disease, t, prec, rec, f1])

print("\nChosen thresholds:", chosen_thresholds)
pd.DataFrame(final_rows, columns=["disease","threshold","precision","recall","f1"]).to_csv(
    f"{DRIVE_DIR}/model_summary_disease_tuned.csv", index=False)

joblib.dump(disease_scaler, "/content/scaler_disease.pkl")
for d, m in disease_models.items():
    joblib.dump(m, f"/content/model_{d.lower()}.pkl")
print("\nSaved scaler_disease.pkl and model_<disease>.pkl for each disease.")

## 7. Export to the Android binary format

Same tree-walking binary format used since v5/v6 (see `RandomForestModel.kt`): `int32 numTrees, numFeatures, numClasses`, then each tree's flattened node arrays. Reused for all four models here — each is just a small independent forest.

In [ ]:
import struct

def export_bin(model, out_path):
    n_classes = model.n_classes_
    n_features = model.n_features_in_
    with open(out_path, "wb") as f:
        f.write(struct.pack("<iii", len(model.estimators_), n_features, n_classes))
        for est in model.estimators_:
            tree = est.tree_
            n_nodes = tree.node_count
            f.write(struct.pack("<i", n_nodes))
            for i in range(n_nodes):
                feature = int(tree.feature[i])
                threshold = float(tree.threshold[i])
                left = int(tree.children_left[i])
                right = int(tree.children_right[i])
                counts = tree.value[i][0]
                total = counts.sum()
                probs = (counts / total) if total > 0 else np.zeros(n_classes)
                f.write(struct.pack("<ifii", feature, threshold, left, right))
                f.write(struct.pack(f"<{n_classes}f", *probs.astype(np.float32)))
    size_kb = os.path.getsize(out_path) / 1024
    print(f"{out_path}: {len(model.estimators_)} trees, "
          f"{sum(e.tree_.node_count for e in model.estimators_)} nodes, {size_kb:.1f} KB")

os.makedirs("/content/bin_out", exist_ok=True)
export_bin(leaf_model, "/content/bin_out/leaf_detector.bin")
export_bin(disease_models["Cordana"], "/content/bin_out/cordana.bin")
export_bin(disease_models["Pestalotiopsis"], "/content/bin_out/pestalotiopsis.bin")
export_bin(disease_models["Sigatoka"], "/content/bin_out/sigatoka.bin")

In [ ]:
# Sanity check: does the exported binary format round-trip to the exact same
# probabilities sklearn itself produces? Re-implements the same tree-walk
# RandomForestModel.kt does, in Python, and compares against predict_proba().

def load_bin(path):
    with open(path, "rb") as f:
        data = f.read()
    off = 0
    numTrees, numFeatures, numClasses = struct.unpack_from("<iii", data, off); off += 12
    trees = []
    for _ in range(numTrees):
        (numNodes,) = struct.unpack_from("<i", data, off); off += 4
        feature=[]; threshold=[]; left=[]; right=[]; probs=[]
        for i in range(numNodes):
            f_, th, l_, r_ = struct.unpack_from("<ifii", data, off); off += 16
            p = struct.unpack_from(f"<{numClasses}f", data, off); off += 4*numClasses
            feature.append(f_); threshold.append(th); left.append(l_); right.append(r_); probs.append(p)
        trees.append((feature, threshold, left, right, probs))
    return numTrees, numFeatures, numClasses, trees

def classify_bin(trees, numTrees, numClasses, x):
    avg = np.zeros(numClasses)
    for feature, threshold, left, right, probs in trees:
        node = 0
        while feature[node] != -2:
            node = left[node] if x[feature[node]] <= threshold[node] else right[node]
        avg += np.array(probs[node])
    return avg / numTrees

def verify(model, scaler, X_test_s, bin_path, name):
    sk_proba = model.predict_proba(X_test_s)
    numTrees, numFeatures, numClasses, trees = load_bin(bin_path)
    max_diff, mism = 0.0, 0
    for i in range(len(X_test_s)):
        p = classify_bin(trees, numTrees, numClasses, X_test_s[i])
        max_diff = max(max_diff, np.max(np.abs(p - sk_proba[i])))
        if np.argmax(p) != np.argmax(sk_proba[i]):
            mism += 1
    print(f"{name}: max_diff={max_diff:.2e}  mismatches={mism}/{len(X_test_s)}")

verify(leaf_model, leaf_scaler, X_val_s if False else leaf_scaler.transform(X_test), "/content/bin_out/leaf_detector.bin", "leaf_detector")
for d in DISEASE_LABELS:
    verify(disease_models[d], disease_scaler, disease_scaler.transform(X_test), f"/content/bin_out/{d.lower()}.bin", d)

In [ ]:
print("MEAN =", ", ".join(repr(float(v)) for v in leaf_scaler.mean_))
print()
print("SCALE =", ", ".join(repr(float(v)) for v in leaf_scaler.scale_))
print()
print("--- disease scaler ---")
print("MEAN =", ", ".join(repr(float(v)) for v in disease_scaler.mean_))
print()
print("SCALE =", ", ".join(repr(float(v)) for v in disease_scaler.scale_))

## 8. Download everything

Copy `bin_out/*.bin` into `app/src/main/assets/`, and paste the printed `MEAN`/`SCALE` arrays above into `FeatureScaler.kt` (`LeafFeatureScaler` / `DiseaseFeatureScaler`) if you retrained — see the README's "Regenerating the `.bin` files if you retrain" section.

In [ ]:
from google.colab import files as gfiles

for fname in ["model_leaf.pkl", "scaler_leaf.pkl", "model_cordana.pkl", "model_pestalotiopsis.pkl",
              "model_sigatoka.pkl", "scaler_disease.pkl"]:
    gfiles.download(f"/content/{fname}")

for fname in ["leaf_detector.bin", "cordana.bin", "pestalotiopsis.bin", "sigatoka.bin"]:
    gfiles.download(f"/content/bin_out/{fname}")

## 9. Inference demo

Runs the full pipeline on a single image path: leaf gate first, then all three disease checks independently.

In [ ]:
def predict(path, leaf_threshold=LEAF_THRESHOLD, thresholds=None):
    thresholds = thresholds or chosen_thresholds
    leaf_feats = extract_leaf_features(path)
    if leaf_feats is None:
        return {"error": "could not read image"}

    leaf_feats_s = leaf_scaler.transform([leaf_feats])[0]
    leaf_prob = leaf_model.predict_proba([leaf_feats_s])[0][1]
    if leaf_prob < leaf_threshold:
        return {"is_leaf": False, "leaf_probability": float(leaf_prob)}

    disease_feats = leaf_feats[:15]
    disease_feats_s = disease_scaler.transform([disease_feats])[0]

    probs = {}
    detected = []
    for disease in DISEASE_LABELS:
        p = disease_models[disease].predict_proba([disease_feats_s])[0][1]
        probs[disease] = float(p)
        if p >= thresholds[disease]:
            detected.append(disease)

    return {
        "is_leaf": True,
        "leaf_probability": float(leaf_prob),
        "diseases_detected": detected,
        "all_probabilities": probs,
        "healthy": len(detected) == 0
    }

# Example:
# sample_path = glob.glob(os.path.join(DATASET_DIR, "MULTI_LABEL", "cordana_pestalotiopsis", "*"))[0]
# predict(sample_path)